# Databricks + Azure OpenAI: Docs Agent
Tool-calling agent (LangGraph) using **Azure OpenAI (gpt-5.4-mini)** as the LLM, with
a TF-IDF document retriever and a TF-IDF keyword tool (Unity Catalog function, with local fallback).
Traced + evaluated with MLflow. No model registration or serving endpoint needed.

In [ ]:
%pip install -U -qqqq mlflow langchain langchain-openai langgraph databricks-langchain "unitycatalog-langchain[databricks]" scikit-learn pandas

In [ ]:
dbutils.library.restartPython()

## 1. Config
Store the key in a Databricks secret scope if you can (`dbutils.secrets.get`).
Otherwise paste it in the widget at runtime. Never hardcode it in the notebook.

In [ ]:
import os

AZURE_OPENAI_BASE_URL = "https://hakunamatata11.openai.azure.com/openai/v1/"
MODEL = "gpt-5.4-mini"  # must match your Azure deployment name

dbutils.widgets.text("azure_openai_key", "", "Azure OpenAI API key")
API_KEY = dbutils.widgets.get("azure_openai_key")
# Alternative: API_KEY = dbutils.secrets.get(scope="<scope>", key="<key>")
assert API_KEY, "Paste your Azure OpenAI API key in the widget at the top"

catalog = "agenticaiengineer"
schema = "default"

## 2. LLM

In [ ]:
from langchain_openai import ChatOpenAI

# gpt-5 family: don't set temperature (only default supported)
llm = ChatOpenAI(
    model=MODEL,
    base_url=AZURE_OPENAI_BASE_URL,
    api_key=API_KEY,
    max_completion_tokens=1000,
)
print(llm.invoke("Reply with one word: ready").content)

## 3. Tools

In [ ]:
from typing import Any
import pandas as pd
import mlflow
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer

docs_url = "https://raw.githubusercontent.com/databricks/genai-cookbook/refs/heads/main/quick_start_demo/chunked_databricks_docs_filtered.jsonl"
documents = pd.read_json(docs_url, lines=True)

doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"])


@tool
@mlflow.trace(name="LittleIndex", span_type=mlflow.entities.SpanType.RETRIEVER)
def find_relevant_documents(query: str, top_n: int = 5) -> list[dict[str, Any]]:
    """Gets relevant Databricks documentation chunks for the query."""
    q = doc_vectorizer.transform([query])
    sims = (tfidf_matrix @ q.T).toarray().flatten()
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)[:top_n]
    return [
        {
            "page_content": documents.iloc[i]["content"],
            "metadata": {"doc_uri": documents.iloc[i]["doc_uri"], "score": float(s)},
        }
        for i, s in ranked
    ]


def tfidf_keywords(text: str) -> list[str]:
    """
    Extracts keywords from the provided text using TF-IDF.

    Args:
        text (string): Input text.
    Returns:
        list[str]: List of extracted keywords in ascending order of importance.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec = TfidfVectorizer(stop_words="english")
    scores = vec.fit_transform([text]).toarray()[0]
    names = vec.get_feature_names_out()
    top = scores.argsort()[-5:][::-1]
    return [names[i] for i in top if scores[i] > 0]


# Unity Catalog function tool (the "Databricks" part). Falls back to a local tool if UC is not permitted.
try:
    from databricks_langchain.uc_ai import (
        DatabricksFunctionClient,
        UCFunctionToolkit,
        set_uc_function_client,
    )

    uc_client = DatabricksFunctionClient()
    set_uc_function_client(uc_client)
    uc_client.create_python_function(func=tfidf_keywords, catalog=catalog, schema=schema, replace=True)
    keyword_tools = UCFunctionToolkit(function_names=[f"{catalog}.{schema}.tfidf_keywords"]).tools
    print("Using Unity Catalog keyword tool")
except Exception as e:
    print(f"UC tool unavailable ({type(e).__name__}), using local tool instead")
    keyword_tools = [tool(tfidf_keywords)]

tools = [*keyword_tools, find_relevant_documents]

## 4. Agent

In [ ]:
from langgraph.prebuilt import create_react_agent

mlflow.langchain.autolog()

SYSTEM_PROMPT = """You are a helpful assistant that answers questions about Databricks.
Questions unrelated to Databricks are irrelevant; politely decline them.
Use the tools: search the docs with find_relevant_documents and base your answer on what it returns.
Use the keyword tool when the user asks for keywords. Ask a follow-up question if the request is unclear."""

agent = create_react_agent(llm, tools=tools, prompt=SYSTEM_PROMPT)


def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    msgs = result["messages"]
    tools_used = [m.name for m in msgs if m.type == "tool"]
    return msgs[-1].content, tools_used


answer, used = ask("What are the keywords for: 'the quick brown fox jumped over the lazy brown dog'?")
print(answer, used)

answer, used = ask("What is Delta Live Tables (DLT) in Databricks?")
print(answer, used)

## 5. Evaluate (no serving or registration needed)
Simple test set with two checks: did the agent call the right tool, and did the answer cover the expected terms.
Results are logged to MLflow as metrics + a table.

In [ ]:
eval_set = [
    {"q": "What is DLT in Databricks?", "expect_tool": "find_relevant_documents", "expect_terms": ["pipeline"]},
    {"q": "What is Unity Catalog?", "expect_tool": "find_relevant_documents", "expect_terms": ["governance"]},
    {"q": "What API lets me parallelize operations over rows of a delta table?", "expect_tool": "find_relevant_documents", "expect_terms": ["spark"]},
    {"q": "Give me keywords for: 'Delta Lake provides ACID transactions for big data workloads'", "expect_tool": "tfidf_keywords", "expect_terms": ["delta"]},
    {"q": "Who won the last cricket world cup?", "expect_tool": None, "expect_terms": []},  # should be declined, no retrieval
]

rows = []
with mlflow.start_run(run_name="azure_openai_docs_agent_eval"):
    for item in eval_set:
        answer, used = ask(item["q"])
        used_names = " ".join(used)
        if item["expect_tool"]:
            tool_ok = item["expect_tool"] in used_names
        else:
            tool_ok = "find_relevant_documents" not in used_names
        terms_ok = all(t.lower() in answer.lower() for t in item["expect_terms"])
        rows.append({"question": item["q"], "answer": answer, "tools_used": used, "tool_ok": tool_ok, "terms_ok": terms_ok})

    results = pd.DataFrame(rows)
    mlflow.log_metric("tool_accuracy", results["tool_ok"].mean())
    mlflow.log_metric("answer_term_accuracy", results["terms_ok"].mean())
    mlflow.log_table(results.astype(str), "eval_results.json")

display(results)
print(f"Tool accuracy: {results['tool_ok'].mean():.0%} | Answer-term accuracy: {results['terms_ok'].mean():.0%}")

## Conclusion
Built a tool-calling agent on Databricks (LangGraph + MLflow tracing + Unity Catalog function)
that consumes **Azure OpenAI gpt-5.4-mini**, combines TF-IDF doc retrieval with a keyword tool,
and evaluated tool usage and answer quality, with results logged to MLflow.
Model registration/serving was skipped because it needs workspace permissions not available here.